# Dataset 8: Investor Transactions — Data Cleaning

## Objective

Inspect, clean, and validate the investor transaction dataset to prepare it for reliable analysis in the Mutual Fund Analytics project.

## Source Dataset

* **Raw file:** `data/raw/08_investor_transactions.csv`
* **Cleaned file:** `data/processed/cleaned_investor_transactions.csv`

## Workflow

1. Load the raw dataset.
2. Inspect the shape, columns, data types, and sample records.
3. Check missing values, duplicates, and inconsistent text.
4. Validate transaction dates and numeric fields.
5. Apply appropriate cleaning rules without changing valid source data unnecessarily.
6. Export the cleaned dataset.
7. Reload the exported file and verify its quality.


In [1]:
import pandas as pd
from pathlib import Path

# Locate the project folder
project_path = Path.cwd().parent

# Load the raw investor transactions dataset
transactions_file = (
    project_path / "data" / "raw" / "08_investor_transactions.csv"
)

transactions_df = pd.read_csv(transactions_file)

print("Dataset loaded successfully!")
print("Dataset shape:", transactions_df.shape)

print("\nColumn names:")
print(transactions_df.columns.tolist())

print("\nData types:")
print(transactions_df.dtypes)

print("\nFirst 5 rows:")
display(transactions_df.head())

Dataset loaded successfully!
Dataset shape: (32778, 13)

Column names:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Data types:
investor_id               str
transaction_date          str
amfi_code               int64
transaction_type          str
amount_inr              int64
state                     str
city                      str
city_tier                 str
age_group                 str
gender                    str
annual_income_lakh    float64
payment_mode              str
kyc_status                str
dtype: object

First 5 rows:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [2]:
# Check missing values in each column
print("Missing values by column:")
print(transactions_df.isnull().sum())

print("\nTotal missing values:")
print(transactions_df.isnull().sum().sum())

# Check duplicate records
print("\nDuplicate rows:", transactions_df.duplicated().sum())

# Display rows containing missing values
missing_rows = transactions_df[
    transactions_df.isnull().any(axis=1)
]

print("\nRows containing missing values:", len(missing_rows))
display(missing_rows.head(10))

# Check unique values in categorical columns
categorical_columns = transactions_df.select_dtypes(
    include=["object", "str"]
).columns

print("\nUnique values in categorical columns:")
for col in categorical_columns:
    print(f"{col}: {transactions_df[col].nunique()} unique values")

Missing values by column:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Total missing values:
0

Duplicate rows: 0

Rows containing missing values: 0


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status



Unique values in categorical columns:
investor_id: 5000 unique values
transaction_date: 516 unique values
transaction_type: 3 unique values
state: 12 unique values
city: 24 unique values
city_tier: 2 unique values
age_group: 5 unique values
gender: 2 unique values
payment_mode: 4 unique values
kyc_status: 2 unique values


In [3]:
# Check empty strings and whitespace-only values
print("Empty or whitespace-only values by column:")

text_columns = transactions_df.select_dtypes(
    include=["object", "str"]
).columns

for col in text_columns:
    empty_count = (
        transactions_df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    if empty_count > 0:
        print(f"{col}: {empty_count}")

# Inspect the first few rows with empty text fields
empty_mask = pd.Series(False, index=transactions_df.index)

for col in text_columns:
    empty_mask |= (
        transactions_df[col]
        .astype("string")
        .str.strip()
        .eq("")
    ).fillna(False)

empty_rows = transactions_df.loc[empty_mask]

print("\nRows containing empty text fields:", len(empty_rows))
display(empty_rows.head(10))

Empty or whitespace-only values by column:

Rows containing empty text fields: 0


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [4]:
# Inspect categorical values
categorical_columns = [
    "transaction_type",
    "state",
    "city_tier",
    "age_group",
    "gender",
    "payment_mode",
    "kyc_status",
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(transactions_df[col].value_counts(dropna=False))

# Inspect numeric columns
numeric_columns = [
    "amount_inr",
    "annual_income_lakh",
]

print("\nNumeric summary:")
display(transactions_df[numeric_columns].describe().T)

# Check for negative transaction amounts
print(
    "\nNegative transaction amounts:",
    (transactions_df["amount_inr"] < 0).sum()
)

# Check for zero transaction amounts
print(
    "Zero transaction amounts:",
    (transactions_df["amount_inr"] == 0).sum()
)

# Check for negative annual incomes
print(
    "Negative annual incomes:",
    (transactions_df["annual_income_lakh"] < 0).sum()
)


transaction_type:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

state:
state
Punjab            2965
Madhya Pradesh    2931
Tamil Nadu        2806
Gujarat           2780
West Bengal       2748
Haryana           2736
Telangana         2718
Uttar Pradesh     2695
Delhi             2677
Karnataka         2621
Rajasthan         2577
Maharashtra       2524
Name: count, dtype: int64

city_tier:
city_tier
T30    21719
B30    11059
Name: count, dtype: int64

age_group:
age_group
26-35    13463
36-45     8146
18-25     4916
46-55     3779
56+       2474
Name: count, dtype: int64

gender:
gender
Male      21809
Female    10969
Name: count, dtype: int64

payment_mode:
payment_mode
Net Banking    8250
Cheque         8228
UPI            8154
Mandate        8146
Name: count, dtype: int64

kyc_status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Numeric summary:


,count,mean,std,min,25%,50%,75%,max
amount_inr,32778.0,107437.318628,150415.905084,400.0,3153.0,17782.5,189324.25,597498.0
annual_income_lakh,32778.0,26.181219,20.805425,3.0,10.6,19.7,37.40,99.7



Negative transaction amounts: 0
Zero transaction amounts: 0
Negative annual incomes: 0


In [5]:
# Check transaction date validity
parsed_dates = pd.to_datetime(
    transactions_df["transaction_date"],
    errors="coerce"
)

print("Invalid transaction dates:", parsed_dates.isna().sum())
print("Earliest transaction date:", parsed_dates.min())
print("Latest transaction date:", parsed_dates.max())

# Check AMFI codes
print("\nMissing or invalid AMFI codes:")
print("Missing codes:", transactions_df["amfi_code"].isna().sum())
print("Non-positive codes:", (transactions_df["amfi_code"] <= 0).sum())

# Check categorical values
expected_categories = {
    "transaction_type": {"SIP", "Lumpsum", "Redemption"},
    "city_tier": {"T30", "B30"},
    "gender": {"Male", "Female"},
    "kyc_status": {"Verified", "Pending"},
}

for col, expected in expected_categories.items():
    actual = set(transactions_df[col].dropna().unique())
    unexpected = actual - expected

    print(f"\n{col}:")
    print("Unexpected values:", unexpected)

Invalid transaction dates: 0
Earliest transaction date: 2024-01-01 00:00:00
Latest transaction date: 2025-05-30 00:00:00

Missing or invalid AMFI codes:
Missing codes: 0
Non-positive codes: 0

transaction_type:
Unexpected values: set()

city_tier:
Unexpected values: set()

gender:
Unexpected values: set()

kyc_status:
Unexpected values: set()


In [6]:
# Inspect all remaining categorical columns
remaining_columns = [
    "state",
    "city",
    "age_group",
    "payment_mode",
]

for col in remaining_columns:
    print(f"\n{col}:")
    print(transactions_df[col].value_counts(dropna=False))

# Inspect transaction amounts by transaction type
print("\nTransaction amount summary by type:")
display(
    transactions_df.groupby("transaction_type")["amount_inr"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(2)
)

# Check for duplicate transaction identifiers
print("\nUnique investor IDs:", transactions_df["investor_id"].nunique())
print("Total transactions:", len(transactions_df))


state:
state
Punjab            2965
Madhya Pradesh    2931
Tamil Nadu        2806
Gujarat           2780
West Bengal       2748
Haryana           2736
Telangana         2718
Uttar Pradesh     2695
Delhi             2677
Karnataka         2621
Rajasthan         2577
Maharashtra       2524
Name: count, dtype: int64

city:
city
Kolkata       2748
Hyderabad     2718
Gurugram      2126
Bhopal        1533
Amritsar      1528
Ahmedabad     1500
Faridabad     1483
Coimbatore    1478
Chandigarh    1437
Indore        1398
Jaipur        1354
Bangalore     1334
Chennai       1328
Mysore        1287
Surat         1280
Udaipur       1223
Mumbai         987
New Delhi      966
Kanpur         936
Lucknow        918
Agra           841
Noida          838
Pune           787
Nashik         750
Name: count, dtype: int64

age_group:
age_group
26-35    13463
36-45     8146
18-25     4916
46-55     3779
56+       2474
Name: count, dtype: int64

payment_mode:
payment_mode
Net Banking    8250
Cheque         8228

,count,min,median,mean,max
transaction_type,,,,,
Lumpsum,8095,4541,251770.0,254456.02,594649
Redemption,4967,4550,247511.0,250558.79,597498
SIP,19716,400,5139.0,11018.13,59998



Unique investor IDs: 5000
Total transactions: 32778


In [7]:
# Step 10: Clean column names, text fields, and transaction dates

# Standardize column names
transactions_df.columns = (
    transactions_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove leading/trailing whitespace from text columns
text_columns = transactions_df.select_dtypes(include="object").columns

for col in text_columns:
    transactions_df[col] = transactions_df[col].str.strip()

# Convert transaction_date to datetime
transactions_df["transaction_date"] = pd.to_datetime(
    transactions_df["transaction_date"],
    errors="raise"
)

# Verify the changes
print("Dataset shape:", transactions_df.shape)
print("\nTransaction date dtype:", transactions_df["transaction_date"].dtype)
print("\nRemaining missing values:", transactions_df.isna().sum().sum())
print("Duplicate rows:", transactions_df.duplicated().sum())

display(transactions_df.head())

Dataset shape: (32778, 13)

Transaction date dtype: datetime64[us]

Remaining missing values: 0
Duplicate rows: 0


C:\Users\HP\AppData\Local\Temp\ipykernel_35984\2715084206.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = transactions_df.select_dtypes(include="object").columns


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [8]:
# Step 11: Investigate suspicious and blank text values

# Inspect the suspicious record
display(transactions_df.iloc[[4]].T)

# Count empty or whitespace-only values in text columns
text_columns = transactions_df.select_dtypes(include="str").columns

empty_counts = (
    transactions_df[text_columns]
    .apply(lambda col: col.str.strip().eq(""))
    .sum()
)

print("Empty or whitespace-only values by column:")
print(empty_counts[empty_counts > 0])

# Inspect records with a suspicious state value
print("\nRecords where state is 'De':")
display(transactions_df[transactions_df["state"].eq("De")])

,4
investor_id,INV004691
transaction_date,2024-01-01 00:00:00
amfi_code,119094
transaction_type,Lumpsum
amount_inr,8682
state,Delhi
city,Noida
city_tier,T30
age_group,26-35
gender,Male


Empty or whitespace-only values by column:
Series([], dtype: int64)

Records where state is 'De':


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [9]:
# Step 12: Final validation

print("FINAL DATA QUALITY REPORT")
print("-" * 35)

print("Rows:", len(transactions_df))
print("Columns:", len(transactions_df.columns))
print("Duplicate rows:", transactions_df.duplicated().sum())
print("Total missing values:", transactions_df.isna().sum().sum())

# Check text fields for empty strings
text_columns = transactions_df.select_dtypes(include="str").columns

empty_total = (
    transactions_df[text_columns]
    .apply(lambda col: col.str.strip().eq(""))
    .sum()
    .sum()
)

print("Empty text values:", empty_total)

# Validate transaction dates
print("Earliest transaction:", transactions_df["transaction_date"].min())
print("Latest transaction:", transactions_df["transaction_date"].max())

# Validate transaction amounts
print("Non-positive amounts:", (transactions_df["amount_inr"] <= 0).sum())

# Check expected transaction types
print("\nTransaction types:")
print(transactions_df["transaction_type"].value_counts())

print("\nFinal column names:")
print(transactions_df.columns.tolist())

FINAL DATA QUALITY REPORT
-----------------------------------
Rows: 32778
Columns: 13
Duplicate rows: 0
Total missing values: 0
Empty text values: 0
Earliest transaction: 2024-01-01 00:00:00
Latest transaction: 2025-05-30 00:00:00
Non-positive amounts: 0

Transaction types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Final column names:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [10]:
# Step 13: Export cleaned investor transactions

from pathlib import Path

project_path = Path.cwd().parent
processed_folder = project_path / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

output_file = processed_folder / "cleaned_investor_transactions.csv"

transactions_df.to_csv(output_file, index=False)

print("Dataset exported successfully!")
print("Saved to:", output_file)
print("Rows exported:", len(transactions_df))
print("Columns exported:", len(transactions_df.columns))
print("File size (KB):", round(output_file.stat().st_size / 1024, 2))

Dataset exported successfully!
Saved to: d:\Mutual-Fund-Analytics\data\processed\cleaned_investor_transactions.csv
Rows exported: 32778
Columns exported: 13
File size (KB): 3086.62


In [11]:
# Step 14: Verify the exported CSV

verified_df = pd.read_csv(output_file)

print("EXPORTED FILE VERIFICATION")
print("-" * 35)

print("Rows:", verified_df.shape[0])
print("Columns:", verified_df.shape[1])
print("Duplicate rows:", verified_df.duplicated().sum())
print("Missing values:", verified_df.isna().sum().sum())

print("\nColumn names:")
print(verified_df.columns.tolist())

print("\nFirst 5 rows:")
display(verified_df.head())

print("\nLast 5 rows:")
display(verified_df.tail())

EXPORTED FILE VERIFICATION
-----------------------------------
Rows: 32778
Columns: 13
Duplicate rows: 0
Missing values: 0

Column names:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

First 5 rows:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending



Last 5 rows:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
32773,INV003340,2025-05-30,101207,Lumpsum,168029,Madhya Pradesh,Indore,T30,26-35,Male,22.5,Net Banking,Verified
32774,INV001838,2025-05-30,119093,SIP,2175,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified
32775,INV000074,2025-05-30,120504,SIP,25998,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified
32776,INV002929,2025-05-30,148568,SIP,459,West Bengal,Kolkata,T30,26-35,Male,13.0,Mandate,Verified
32777,INV001852,2025-05-30,118633,Redemption,226721,Telangana,Hyderabad,T30,26-35,Male,21.7,Cheque,Verified


## Dataset 8: Investor Transactions — Cleaning Summary

### Dataset Overview

* **Source file:** `08_investor_transactions.csv`
* **Cleaned file:** `cleaned_investor_transactions.csv`
* **Original and final dimensions:** 32,778 rows × 13 columns

### Cleaning and Validation Performed

* Standardized column names to lowercase and snake_case.
* Removed leading and trailing whitespace from text fields.
* Converted `transaction_date` to datetime format.
* Checked for missing values, empty text fields, and duplicate rows.
* Validated transaction dates and transaction amounts.
* Checked the expected categories for transaction type, city tier, gender, and KYC status.
* Exported the cleaned dataset without an additional CSV index column.
* Reloaded the exported CSV to verify its structure.

### Final Data Quality Results

* Missing values: 0
* Empty text values: 0
* Duplicate rows: 0
* Invalid transaction dates: 0
* Non-positive transaction amounts: 0

### Transaction Distribution

| Transaction Type |      Count |
| ---------------- | ---------: |
| SIP              |     19,716 |
| Lumpsum          |      8,095 |
| Redemption       |      4,967 |
| **Total**        | **32,778** |

### Conclusion

The investor transactions dataset passed the performed data-quality checks and was exported for subsequent analysis. Original transaction amounts and records were preserved.
